In [2]:
import os
import io
import cv2
from PIL import Image
from rembg import remove
from PIL import ImageOps
# === 配置路径 ===
input_folder = 'assets/images'
output_folder = 'image2'
target_size = (300, 300)
margin = 0.9  # 裁剪框边距系数
face_cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"

# === 初始化 ===
face_cascade = cv2.CascadeClassifier(face_cascade_path)
os.makedirs(output_folder, exist_ok=True)

# === 批处理图片 ===
for filename in os.listdir(input_folder):
    if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue

    image_path = os.path.join(input_folder, filename)
    img_cv = cv2.imread(image_path)

    if img_cv is None:
        print(f"⚠️ 无法读取 {filename}")
        continue

    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

    if len(faces) == 0:
        print(f"❌ 未检测到人脸：{filename}，跳过")
        continue

    # === 选最大人脸区域 ===
    x, y, w, h = max(faces, key=lambda rect: rect[2] * rect[3])
    r = int(max(w, h) * (1 + margin))
    cx, cy = x + w // 2, y + h // 2

    left = max(0, cx - r // 2)
    top = max(0, cy - r // 2)
    right = min(img_cv.shape[1], cx + r // 2)
    bottom = min(img_cv.shape[0], cy + r // 2)

    face_crop = img_cv[top:bottom, left:right]

    # === Step 1：OpenCV → RGBA for rembg ===
    face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
    face_pil = Image.fromarray(face_rgb).convert("RGBA")

    # === Step 2：抠图去背景 ===
    buffered = io.BytesIO()
    face_pil.save(buffered, format="PNG")
    output_bytes = remove(buffered.getvalue())  # 调用 rembg
    result_rgba = Image.open(io.BytesIO(output_bytes)).convert("RGBA")

    # === Step 3：合成白色背景图像 ===
    white_bg = Image.new("RGBA", result_rgba.size, (255, 255, 255, 255))
    final_img = Image.alpha_composite(white_bg, result_rgba).convert("RGB")

    # === Step 4：统一输出尺寸 ===
    # 推荐做法：保持比例缩放 + 白色背景填充成正方形
    # 将图片等比缩放到不超过 target_size
    final_resized = ImageOps.contain(final_img, target_size, method=Image.LANCZOS)

    # 创建白色背景画布并居中粘贴
    canvas = Image.new("RGB", target_size, (255, 255, 255))
    offset = ((target_size[0] - final_resized.width) // 2, (target_size[1] - final_resized.height) // 2)
    canvas.paste(final_resized, offset)
    final_resized = canvas  # 最终结果
    # === Step 5：保存结果 ===
    output_path = os.path.join(output_folder, filename)
    final_resized.save(output_path)
    print(f"✅ 已处理：{filename}")

print("🎉 所有图片处理完成！输出路径:", output_folder)

✅ 已处理：wangshuaijia.jpg
❌ 未检测到人脸：paper.jpg，跳过
✅ 已处理：bianjunhao.jpg
✅ 已处理：jiangxinyu.jpg
✅ 已处理：jxs.jpg
✅ 已处理：zhoutao.jpg
❌ 未检测到人脸：book2.jpg，跳过
❌ 未检测到人脸：book1.jpg，跳过
✅ 已处理：chenyuxi.jpg
✅ 已处理：yangjinmin.jpg
✅ 已处理：luoyuxin.jpg
✅ 已处理：logo.jpg
✅ 已处理：wanshuyan.jpg
✅ 已处理：zhangchi.jpg
✅ 已处理：songqiran.jpg
✅ 已处理：denzhenyu.jpg
✅ 已处理：caoyi.jpg
✅ 已处理：biyilin.jpg
🎉 所有图片处理完成！输出路径: image2
